# Exploratory Data Analysis — Whisper Fine-Tuning Pipeline

Explore audio datasets for low-resource ASR languages (Amharic, Afaan Oromo):
- Dataset overview and sample counts
- Audio duration distributions
- Vocabulary analysis
- Speaker statistics
- Sample spectrograms

In [ ]:
import sys
from pathlib import Path

import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from datasets import load_from_disk

PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from preprocessing.dataset_builder import DatasetBuilder

sns.set_theme(style="whitegrid")
DATASET_PATH = PROJECT_ROOT / "data" / "processed" / "hf_dataset"
RAW_DIR = PROJECT_ROOT / "data" / "raw"

## 1. Load Dataset

In [ ]:
if DATASET_PATH.exists():
    dataset = load_from_disk(str(DATASET_PATH))
    print(f"Splits: {list(dataset.keys())}")
    for split, ds in dataset.items():
        print(f"  {split}: {len(ds)} samples")
else:
    print("Dataset not found. Run preprocessing first:")
    print("  python scripts/run_preprocess.py --config config/default.yaml")

## 2. Audio Duration Distribution

In [ ]:
def collect_durations(ds):
    durations = []
    for item in ds:
        audio = item["audio"]
        durations.append(len(audio["array"]) / audio["sampling_rate"])
    return durations

if DATASET_PATH.exists():
    train_ds = dataset["train"]
    durations = collect_durations(train_ds)

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    axes[0].hist(durations, bins=30, color="steelblue", edgecolor="white")
    axes[0].set_xlabel("Duration (seconds)")
    axes[0].set_ylabel("Count")
    axes[0].set_title("Audio Duration Distribution")

    axes[1].boxplot(durations, vert=True)
    axes[1].set_ylabel("Duration (seconds)")
    axes[1].set_title("Duration Box Plot")

    plt.tight_layout()
    plt.show()
    print(f"Mean: {np.mean(durations):.2f}s | Median: {np.median(durations):.2f}s | Total: {sum(durations)/3600:.2f}h")

## 3. Vocabulary Analysis

In [ ]:
if DATASET_PATH.exists():
    texts = [item["text"] for item in train_ds]
    words = []
    for t in texts:
        words.extend(t.split())

    word_counts = pd.Series(words).value_counts().head(30)

    fig, ax = plt.subplots(figsize=(12, 5))
    word_counts.plot(kind="bar", ax=ax, color="coral")
    ax.set_title("Top 30 Words in Training Set")
    ax.set_xlabel("Word")
    ax.set_ylabel("Frequency")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

    char_lengths = [len(t) for t in texts]
    print(f"Unique words: {len(set(words))} | Avg chars/transcript: {np.mean(char_lengths):.1f}")

## 4. Speaker Statistics

In [ ]:
if DATASET_PATH.exists():
    speakers = [item.get("speaker", "unknown") for item in train_ds]
    speaker_counts = pd.Series(speakers).value_counts()

    fig, ax = plt.subplots(figsize=(10, 4))
    speaker_counts.head(20).plot(kind="barh", ax=ax, color="seagreen")
    ax.set_title("Samples per Speaker (Top 20)")
    ax.set_xlabel("Sample Count")
    plt.tight_layout()
    plt.show()
    print(f"Unique speakers: {len(speaker_counts)}")

## 5. Sample Spectrograms

In [ ]:
if DATASET_PATH.exists():
    n_samples = min(4, len(train_ds))
    fig, axes = plt.subplots(n_samples, 1, figsize=(12, 3 * n_samples))
    if n_samples == 1:
        axes = [axes]

    for i in range(n_samples):
        item = train_ds[i]
        audio = item["audio"]["array"]
        sr = item["audio"]["sampling_rate"]
        mel = librosa.feature.melspectrogram(y=audio, sr=sr, n_mels=128)
        mel_db = librosa.power_to_db(mel, ref=np.max)
        librosa.display.specshow(mel_db, sr=sr, x_axis="time", y_axis="mel", ax=axes[i])
        axes[i].set_title(f"Sample {i}: {item['text'][:50]}")

    plt.tight_layout()
    plt.show()

## 6. Dataset Statistics Export

In [ ]:
if DATASET_PATH.exists():
    builder = DatasetBuilder(audio_dir=RAW_DIR)
    stats = builder.compute_statistics(dataset)
    print("Dataset Statistics:")
    for k, v in stats.to_dict().items():
        print(f"  {k}: {v}")